#**Connect Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#**Load Dataset**

In [2]:
from pathlib import Path

DATA_DIR = Path("/content/APTOS")
CSV_PATH = DATA_DIR / "train.csv"
IMAGE_DIR = DATA_DIR / "train_images"

# Check whether dataset is already fully available
image_count = len(list(IMAGE_DIR.glob("*.png"))) if IMAGE_DIR.exists() else 0

if CSV_PATH.exists() and image_count == 3662:
    print("APTOS already ready.")
    print("CSV:", CSV_PATH)
    print("Images:", image_count)
    print("Skipping download and unzip.")

else:
    print("APTOS not complete. Downloading...")

    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json

    # Download only if ZIP does not already exist
    if not (DATA_DIR / "aptos2019-blindness-detection.zip").exists():
        !kaggle competitions download -c aptos2019-blindness-detection -p /content/APTOS

    # -o = overwrite automatically, NO questions
    !unzip -q -o /content/APTOS/aptos2019-blindness-detection.zip -d /content/APTOS

    image_count = len(list(IMAGE_DIR.glob("*.png")))

    print("CSV:", CSV_PATH.exists())
    print("Images:", image_count)

APTOS not complete. Downloading...
100% 9.51G/9.51G [08:40<00:00, 19.6MB/s]

CSV: True
Images: 3662


#**DINOv2-Large APTOS embedding extraction**

In [3]:
# ============================================================
# APTOS -> DINOv2-Large Frozen Embeddings
#
# SAME APTOS PIPELINE AS RETFOUND / MEDSIGLIP
#
# Same:
#   - raw APTOS images
#   - exact duplicate audit
#   - 3504 KEEP cases
#   - image order
#   - black-border removal, threshold = 7
#   - aspect-ratio preservation
#   - square padding
#
# DINOv2-specific:
#   - official AutoImageProcessor
#   - frozen DINOv2-Large encoder
#   - CLS token, 1024-D embedding
#
# Saves ONLY:
#   aptos_dinov2_large_embeddings.npz
# ============================================================

import numpy as np
import pandas as pd
import torch

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, AutoModel


# ============================================================
# 1. PATHS
# ============================================================

MODEL_ID = "facebook/dinov2-large"

IMAGE_DIR = Path(
    "/content/APTOS/train_images"
)

AUDIT_PATH = Path(
    "/content/drive/MyDrive/thesis_DR/aptos_exact_duplicate_audit.csv"
)

SAVE_PATH = Path(
    "/content/drive/MyDrive/thesis_DR/aptos_dinov2_large_embeddings.npz"
)

BATCH_SIZE = 8


# ============================================================
# 2. LOAD EXACT SAME 3504 APTOS KEEP CASES
# ============================================================

audit = pd.read_csv(
    AUDIT_PATH,
    dtype={"image_id": str}
)

keep_df = (
    audit[
        audit["action"] == "KEEP"
    ][
        [
            "image_id",
            "original_grade",
            "binary_label"
        ]
    ]
    .sort_values("image_id")
    .reset_index(drop=True)
)

print("APTOS KEEP images:", len(keep_df))

assert len(keep_df) == 3504


# ============================================================
# 3. CONFIRM RAW IMAGES
# ============================================================

missing = [
    image_id
    for image_id in keep_df["image_id"]
    if not (
        IMAGE_DIR /
        f"{image_id}.png"
    ).exists()
]

assert len(missing) == 0, (
    f"{len(missing)} raw APTOS images missing. "
    "Rerun the original APTOS dataset-loading cell."
)

print("Raw image check: PASS")


# ============================================================
# 4. SAME CUSTOM RETINAL PREPROCESSING
# ============================================================

def remove_black_border(
    image,
    threshold=7
):

    img = np.asarray(image)

    gray = (
        0.299 * img[:, :, 0]
        +
        0.587 * img[:, :, 1]
        +
        0.114 * img[:, :, 2]
    )

    mask = gray > threshold

    if not mask.any():
        return image

    rows = np.where(
        mask.any(axis=1)
    )[0]

    cols = np.where(
        mask.any(axis=0)
    )[0]

    cropped = img[
        rows[0]:rows[-1] + 1,
        cols[0]:cols[-1] + 1
    ]

    return Image.fromarray(
        cropped
    )


def pad_to_square(image):

    width, height = image.size

    side = max(
        width,
        height
    )

    square = Image.new(
        "RGB",
        (side, side),
        (0, 0, 0)
    )

    left = (
        side - width
    ) // 2

    top = (
        side - height
    ) // 2

    square.paste(
        image,
        (left, top)
    )

    return square


# ============================================================
# 5. DATASET
# ============================================================

class APTOSDINOv2Dataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_dir
    ):

        self.df = dataframe
        self.image_dir = image_dir


    def __len__(self):

        return len(self.df)


    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        image_id = str(
            row["image_id"]
        )

        image_path = (
            self.image_dir /
            f"{image_id}.png"
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        # ----------------------------------------
        # SAME custom preprocessing
        # ----------------------------------------

        image = remove_black_border(
            image,
            threshold=7
        )

        image = pad_to_square(
            image
        )

        return (
            image,
            image_id,
            int(
                row["original_grade"]
            ),
            int(
                row["binary_label"]
            )
        )


# ============================================================
# 6. DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ============================================================
# 7. LOAD OFFICIAL DINOv2 PROCESSOR
# ============================================================

processor = AutoImageProcessor.from_pretrained(
    MODEL_ID
)


# ============================================================
# 8. LOAD FROZEN DINOv2-LARGE
# ============================================================

if device.type == "cuda":

    model = AutoModel.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )

else:

    model = AutoModel.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    )


for parameter in model.parameters():
    parameter.requires_grad = False


model = model.to(
    device
)

model.eval()


print("DINOv2-Large loaded.")

print(
    "Encoder frozen:",
    not any(
        p.requires_grad
        for p in model.parameters()
    )
)

print(
    "Hidden dimension:",
    model.config.hidden_size
)

assert model.config.hidden_size == 1024


# ============================================================
# 9. DATASET / LOADER
# ============================================================

dataset = APTOSDINOv2Dataset(
    keep_df,
    IMAGE_DIR
)


loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=lambda batch: batch,
    pin_memory=False
)


# ============================================================
# 10. EXTRACT DINOv2 CLS EMBEDDINGS
# ============================================================

all_embeddings = []

all_image_ids = []

all_grades = []

all_binary_labels = []


with torch.inference_mode():

    for batch in tqdm(
        loader,
        desc="Extracting DINOv2-Large embeddings"
    ):

        images = [
            item[0]
            for item in batch
        ]

        image_ids = [
            item[1]
            for item in batch
        ]

        grades = [
            item[2]
            for item in batch
        ]

        labels = [
            item[3]
            for item in batch
        ]


        # ----------------------------------------
        # DINOv2 native resize / crop /
        # normalization
        # ----------------------------------------

        inputs = processor(
            images=images,
            return_tensors="pt"
        )


        pixel_values = inputs[
            "pixel_values"
        ].to(
            device=device,
            dtype=next(
                model.parameters()
            ).dtype
        )


        outputs = model(
            pixel_values=pixel_values
        )


        # ----------------------------------------
        # Official downstream whole-image feature
        #
        # CLS token = first token
        # Shape:
        # [batch, 1024]
        # ----------------------------------------

        features = (
            outputs
            .last_hidden_state[
                :,
                0,
                :
            ]
        )


        features = (
            features
            .float()
            .cpu()
            .numpy()
        )


        all_embeddings.append(
            features
        )

        all_image_ids.extend(
            image_ids
        )

        all_grades.extend(
            grades
        )

        all_binary_labels.extend(
            labels
        )


# ============================================================
# 11. COMBINE
# ============================================================

embeddings = np.concatenate(
    all_embeddings,
    axis=0
).astype(
    np.float32
)


image_ids = np.asarray(
    all_image_ids
)


grades = np.asarray(
    all_grades,
    dtype=np.int64
)


binary_labels = np.asarray(
    all_binary_labels,
    dtype=np.int64
)


# ============================================================
# 12. STRICT VALIDATION
# ============================================================

print(
    "\nEmbedding shape:",
    embeddings.shape
)


assert embeddings.shape == (
    3504,
    1024
)


assert len(
    image_ids
) == 3504


assert len(
    np.unique(
        image_ids
    )
) == 3504


# Exact same order as audit KEEP table
assert (
    image_ids.tolist()
    ==
    keep_df[
        "image_id"
    ].astype(str).tolist()
)


# Labels must remain aligned
assert (
    grades
    ==
    keep_df[
        "original_grade"
    ].to_numpy(
        dtype=np.int64
    )
).all()


assert (
    binary_labels
    ==
    keep_df[
        "binary_label"
    ].to_numpy(
        dtype=np.int64
    )
).all()


assert np.isfinite(
    embeddings
).all()


# ============================================================
# 13. SAVE ONE PERMANENT FILE
# ============================================================

np.savez_compressed(
    SAVE_PATH,

    embeddings=
        embeddings,

    image_id=
        image_ids,

    original_grade=
        grades,

    binary_label=
        binary_labels
)


# ============================================================
# 14. FINAL REPORT
# ============================================================

print(
    "\n========== DINOV2-LARGE EMBEDDING EXTRACTION =========="
)

print(
    "\nModel:"
)

print(
    "DINOv2-Large"
)

print(
    "Model ID:",
    MODEL_ID
)

print(
    "Encoder status: FROZEN"
)


print(
    "\nShared retinal preprocessing:"
)

print(
    "1. Remove black outer border, threshold = 7"
)

print(
    "2. Preserve aspect ratio"
)

print(
    "3. Pad to square"
)


print(
    "\nModel-specific preprocessing:"
)

print(
    "4. Official DINOv2 AutoImageProcessor"
)


print(
    "\nWhole-image representation:"
)

print(
    "5. Final-layer CLS token"
)


print(
    "\nImages processed:",
    len(
        image_ids
    )
)


print(
    "Embedding shape:",
    embeddings.shape
)


print(
    "NaN values:",
    np.isnan(
        embeddings
    ).sum()
)


print(
    "Infinite values:",
    np.isinf(
        embeddings
    ).sum()
)


print(
    "\nSaved permanently:"
)

print(
    SAVE_PATH
)


print(
    "\n========== COMPLETE =========="
)

APTOS KEEP images: 3504
Raw image check: PASS
Device: cuda


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/549 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

DINOv2-Large loaded.
Encoder frozen: True
Hidden dimension: 1024


Extracting DINOv2-Large embeddings:   0%|          | 0/438 [00:00<?, ?it/s]


Embedding shape: (3504, 1024)

========== DINOV2-LARGE EMBEDDING EXTRACTION ==========

Model:
DINOv2-Large
Model ID: facebook/dinov2-large
Encoder status: FROZEN

Shared retinal preprocessing:
1. Remove black outer border, threshold = 7
2. Preserve aspect ratio
3. Pad to square

Model-specific preprocessing:
4. Official DINOv2 AutoImageProcessor

Whole-image representation:
5. Final-layer CLS token

Images processed: 3504
Embedding shape: (3504, 1024)
NaN values: 0
Infinite values: 0

Saved permanently:
/content/drive/MyDrive/thesis_DR/aptos_dinov2_large_embeddings.npz

========== COMPLETE ==========


#**DINOv2-LARGE MLP TRAINING**

In [4]:
# ============================================================
# DINOv2-LARGE MLP TRAINING
#
# SAME TRAINING PIPELINE AS RETFOUND-GREEN / MEDSIGLIP
#
# Only changes:
#   embedding dimension = 1024
#   DINOv2 filenames
# ============================================================

import random
import copy
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold


# ============================================================
# PATHS
# ============================================================

BASE = Path("/content/drive/MyDrive/thesis_DR")

EMBEDDING_PATH = BASE / "aptos_dinov2_large_embeddings.npz"
SPLIT_PATH = BASE / "aptos_split_manifest.csv"

MODEL_SAVE_PATH = BASE / "aptos_dinov2_large_mlp_heads.pt"
PRED_SAVE_PATH = BASE / "aptos_dinov2_large_raw_predictions.csv"


# ============================================================
# SAME SETTINGS AS OTHER MODELS
# ============================================================

SEEDS = [42, 43, 44, 45, 46]

INPUT_DIM = 1024
HIDDEN_DIM = 512
DROPOUT = 0.10

BATCH_SIZE = 64

MAX_EPOCHS = 25
EARLY_STOP_PATIENCE = 4
MIN_DELTA = 1e-5

INNER_FOLDS = 3

HPO_CONFIGS = [
    {
        "name": "A",
        "learning_rate": 3e-4,
        "weight_decay": 0.0
    },
    {
        "name": "B",
        "learning_rate": 3e-4,
        "weight_decay": 1e-4
    },
    {
        "name": "C",
        "learning_rate": 1e-3,
        "weight_decay": 0.0
    },
    {
        "name": "D",
        "learning_rate": 1e-3,
        "weight_decay": 1e-4
    }
]


# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# SAME MLP HEAD
# ============================================================

class MLPHead(nn.Module):

    def __init__(
        self,
        input_dim=1024,
        hidden_dim=512,
        dropout=0.10
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(1)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# LOAD EMBEDDINGS
# ============================================================

data = np.load(
    EMBEDDING_PATH,
    allow_pickle=False
)

embeddings = data["embeddings"].astype(np.float32)
embedding_ids = data["image_id"].astype(str)
embedding_grades = data["original_grade"].astype(int)
embedding_labels = data["binary_label"].astype(int)

assert embeddings.shape == (3504, 1024)
assert len(np.unique(embedding_ids)) == 3504
assert np.isfinite(embeddings).all()

print("\n========== INPUT CHECK ==========")
print("Embedding shape:", embeddings.shape)
print("Unique image IDs:", len(np.unique(embedding_ids)))


# ============================================================
# LOAD SAME SPLIT MANIFEST
# ============================================================

manifest = pd.read_csv(
    SPLIT_PATH,
    dtype={"image_id": str}
)

assert sorted(
    manifest["seed"].unique().tolist()
) == SEEDS

print("Manifest rows:", len(manifest))
print("Seeds:", SEEDS)


embedding_meta = pd.DataFrame({
    "image_id": embedding_ids,
    "embedding_index": np.arange(len(embedding_ids)),
    "embedding_grade": embedding_grades,
    "embedding_label": embedding_labels
})


# ============================================================
# INNER CV TRAINING
# ============================================================

def train_with_early_stopping(
    X_train,
    y_train,
    X_val,
    y_val,
    learning_rate,
    weight_decay,
    run_seed
):

    set_seed(run_seed)

    model = MLPHead(
        INPUT_DIM,
        HIDDEN_DIM,
        DROPOUT
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    train_dataset = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32)
    )

    generator = torch.Generator()
    generator.manual_seed(run_seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    X_val_tensor = torch.tensor(
        X_val,
        dtype=torch.float32,
        device=device
    )

    y_val_tensor = torch.tensor(
        y_val,
        dtype=torch.float32,
        device=device
    )

    best_val_loss = float("inf")
    best_epoch = 1
    best_state = None
    patience_counter = 0

    train_history = []
    val_history = []

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()

        running_loss = 0.0
        n_seen = 0

        for batch_x, batch_y in train_loader:

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad(set_to_none=True)

            logits = model(batch_x)

            loss = criterion(
                logits,
                batch_y
            )

            loss.backward()
            optimizer.step()

            n = batch_x.size(0)

            running_loss += loss.item() * n
            n_seen += n

        train_loss = running_loss / n_seen

        model.eval()

        with torch.inference_mode():

            val_logits = model(
                X_val_tensor
            )

            val_loss = criterion(
                val_logits,
                y_val_tensor
            ).item()

        train_history.append(float(train_loss))
        val_history.append(float(val_loss))

        if val_loss < best_val_loss - MIN_DELTA:

            best_val_loss = val_loss
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:

            patience_counter += 1

        if patience_counter >= EARLY_STOP_PATIENCE:
            break

    model.load_state_dict(
        best_state
    )

    result = {
        "best_val_loss": float(best_val_loss),
        "best_epoch": int(best_epoch),
        "epochs_run": int(len(train_history)),
        "train_loss_history": train_history,
        "val_loss_history": val_history
    }

    del model
    del X_val_tensor
    del y_val_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# ============================================================
# FINAL DEVELOPMENT TRAINING
# ============================================================

def train_final_head(
    X,
    y,
    learning_rate,
    weight_decay,
    epochs,
    run_seed
):

    set_seed(run_seed)

    model = MLPHead(
        INPUT_DIM,
        HIDDEN_DIM,
        DROPOUT
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32)
    )

    generator = torch.Generator()
    generator.manual_seed(run_seed)

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    loss_history = []

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0
        n_seen = 0

        for batch_x, batch_y in loader:

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad(set_to_none=True)

            logits = model(batch_x)

            loss = criterion(
                logits,
                batch_y
            )

            loss.backward()
            optimizer.step()

            n = batch_x.size(0)

            running_loss += loss.item() * n
            n_seen += n

        loss_history.append(
            float(
                running_loss / n_seen
            )
        )

    return model, loss_history


# ============================================================
# RUN FIVE SEEDS
# ============================================================

saved_heads = {}
prediction_tables = []


print(
    "\n========== FINAL DINOV2-LARGE MLP TRAINING =========="
)

print("\nBackbone: DINOv2-Large")
print("Encoder: FROZEN")

print(
    "\nArchitecture:"
    "\n1024 -> 512 -> ReLU -> Dropout(0.1) -> 1 logit"
)

print(
    "\nHPO:"
    "\n3-fold inner stratified CV"
    "\nStratification: original grades 0-4"
    "\nSelection metric: validation BCE loss"
)

print(
    "\nIMPORTANT:"
    "\nHPO uses DEVELOPMENT data only."
    "\nProbability-calibration data: NOT USED"
    "\nRisk-calibration data: NOT USED"
    "\nInternal-test data: NOT USED"
)


for seed in SEEDS:

    print(
        f"\n\n========== SEED {seed} =========="
    )

    seed_df = (
        manifest[
            manifest["seed"] == seed
        ]
        .copy()
        .merge(
            embedding_meta,
            on="image_id",
            how="left",
            validate="one_to_one"
        )
    )

    assert seed_df[
        "embedding_index"
    ].isna().sum() == 0

    assert (
        seed_df["original_grade"].astype(int).values
        ==
        seed_df["embedding_grade"].astype(int).values
    ).all()

    assert (
        seed_df["binary_label"].astype(int).values
        ==
        seed_df["embedding_label"].astype(int).values
    ).all()


    # --------------------------------------------------------
    # DEVELOPMENT ONLY
    # --------------------------------------------------------

    dev_df = (
        seed_df[
            seed_df["split"] == "development"
        ]
        .copy()
        .reset_index(drop=True)
    )

    assert len(dev_df) == 2102

    dev_indices = (
        dev_df[
            "embedding_index"
        ]
        .astype(int)
        .values
    )

    X_dev = embeddings[
        dev_indices
    ]

    y_dev = (
        dev_df[
            "binary_label"
        ]
        .astype(int)
        .values
    )

    grades_dev = (
        dev_df[
            "original_grade"
        ]
        .astype(int)
        .values
    )

    print(
        "Development images:",
        len(dev_df)
    )

    print(
        "Non-referable:",
        int((y_dev == 0).sum())
    )

    print(
        "Referable:",
        int((y_dev == 1).sum())
    )


    # --------------------------------------------------------
    # INNER CV
    # --------------------------------------------------------

    skf = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=seed
    )

    hpo_results = []


    for config in HPO_CONFIGS:

        fold_results = []


        for fold_number, (
            train_idx,
            val_idx
        ) in enumerate(
            skf.split(
                X_dev,
                grades_dev
            ),
            start=1
        ):

            fold_seed = (
                seed * 100
                +
                fold_number
            )


            result = train_with_early_stopping(
                X_dev[train_idx],
                y_dev[train_idx],
                X_dev[val_idx],
                y_dev[val_idx],
                config["learning_rate"],
                config["weight_decay"],
                fold_seed
            )

            fold_results.append(
                result
            )


        fold_losses = [
            r["best_val_loss"]
            for r in fold_results
        ]

        fold_epochs = [
            r["best_epoch"]
            for r in fold_results
        ]

        mean_val_loss = float(
            np.mean(fold_losses)
        )

        std_val_loss = float(
            np.std(
                fold_losses,
                ddof=1
            )
        )


        hpo_results.append({
            "name": config["name"],
            "learning_rate":
                config["learning_rate"],
            "weight_decay":
                config["weight_decay"],
            "mean_val_loss":
                mean_val_loss,
            "std_val_loss":
                std_val_loss,
            "fold_best_epochs":
                fold_epochs,
            "fold_results":
                fold_results
        })


        print(
            f"\nConfig {config['name']}:"
        )

        print(
            "  Learning rate:",
            config["learning_rate"]
        )

        print(
            "  Weight decay:",
            config["weight_decay"]
        )

        print(
            "  Fold best epochs:",
            fold_epochs
        )

        print(
            "  Mean validation BCE:",
            f"{mean_val_loss:.6f}"
        )

        print(
            "  Std validation BCE:",
            f"{std_val_loss:.6f}"
        )


    # --------------------------------------------------------
    # SELECT BEST CONFIG
    # --------------------------------------------------------

    best_config = min(
        hpo_results,
        key=lambda x:
            x["mean_val_loss"]
    )


    selected_epochs = int(
        np.median(
            best_config[
                "fold_best_epochs"
            ]
        )
    )

    selected_epochs = max(
        1,
        min(
            selected_epochs,
            MAX_EPOCHS
        )
    )


    print(
        "\n---------- SELECTED ----------"
    )

    print(
        "Configuration:",
        best_config["name"]
    )

    print(
        "Learning rate:",
        best_config["learning_rate"]
    )

    print(
        "Weight decay:",
        best_config["weight_decay"]
    )

    print(
        "Mean CV validation BCE:",
        f"{best_config['mean_val_loss']:.6f}"
    )

    print(
        "Fold best epochs:",
        best_config["fold_best_epochs"]
    )

    print(
        "Final training epochs:",
        selected_epochs
    )


    # --------------------------------------------------------
    # FINAL TRAINING
    # --------------------------------------------------------

    final_seed = (
        seed * 1000
        +
        999
    )

    final_model, loss_history = train_final_head(
        X_dev,
        y_dev,
        best_config["learning_rate"],
        best_config["weight_decay"],
        selected_epochs,
        final_seed
    )


    print(
        "Final development training loss:",
        f"{loss_history[-1]:.6f}"
    )


    # --------------------------------------------------------
    # RAW PREDICTIONS FOR ALL CASES
    # --------------------------------------------------------

    all_indices = (
        seed_df[
            "embedding_index"
        ]
        .astype(int)
        .values
    )

    X_all = torch.tensor(
        embeddings[
            all_indices
        ],
        dtype=torch.float32,
        device=device
    )

    final_model.eval()

    with torch.inference_mode():

        raw_logits = final_model(
            X_all
        )

        raw_probabilities = torch.sigmoid(
            raw_logits
        )


    raw_logits = (
        raw_logits
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    raw_probabilities = (
        raw_probabilities
        .cpu()
        .numpy()
        .astype(np.float32)
    )


    pred_df = seed_df[
        [
            "seed",
            "image_id",
            "split",
            "original_grade",
            "binary_label"
        ]
    ].copy()

    pred_df[
        "raw_logit"
    ] = raw_logits

    pred_df[
        "raw_probability"
    ] = raw_probabilities

    prediction_tables.append(
        pred_df
    )


    cpu_state_dict = {
        name:
            tensor.detach().cpu()

        for name, tensor
        in final_model
        .state_dict()
        .items()
    }


    saved_heads[seed] = {

        "state_dict":
            cpu_state_dict,

        "selected_config":
            best_config["name"],

        "selected_learning_rate":
            float(
                best_config[
                    "learning_rate"
                ]
            ),

        "selected_weight_decay":
            float(
                best_config[
                    "weight_decay"
                ]
            ),

        "selected_epochs":
            int(
                selected_epochs
            ),

        "development_images":
            int(len(dev_df)),

        "final_training_loss":
            float(
                loss_history[-1]
            ),

        "final_loss_history":
            loss_history,

        "hpo_results":
            hpo_results
    }


    del final_model
    del X_all

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# COMBINE / VALIDATE
# ============================================================

predictions = pd.concat(
    prediction_tables,
    ignore_index=True
)

expected_rows = (
    3504 * len(SEEDS)
)

assert len(predictions) == expected_rows

assert predictions[
    "raw_logit"
].isna().sum() == 0

assert np.isfinite(
    predictions[
        "raw_logit"
    ].values
).all()


# ============================================================
# SAVE FINAL MODEL BUNDLE
# ============================================================

model_bundle = {

    "backbone":
        "DINOv2-Large",

    "encoder_status":
        "frozen",

    "embedding_dim":
        1024,

    "head_architecture": {
        "input_dim": 1024,
        "hidden_dim": 512,
        "activation": "ReLU",
        "dropout": 0.10,
        "output_dim": 1
    },

    "training_protocol": {
        "loss":
            "BCEWithLogitsLoss",

        "optimizer":
            "AdamW",

        "batch_size":
            BATCH_SIZE,

        "max_epochs":
            MAX_EPOCHS,

        "early_stopping_patience":
            EARLY_STOP_PATIENCE,

        "inner_cv_folds":
            INNER_FOLDS,

        "inner_cv_stratification":
            "original_grade_0_to_4",

        "hpo_selection_metric":
            "mean_inner_validation_BCE",

        "final_epoch_rule":
            "median_best_epoch_of_selected_configuration",

        "class_weighting":
            None
    },

    "hpo_configs":
        HPO_CONFIGS,

    "seeds":
        SEEDS,

    "heads":
        saved_heads
}


torch.save(
    model_bundle,
    MODEL_SAVE_PATH
)


# ============================================================
# SAVE RAW PREDICTIONS
# ============================================================

predictions.to_csv(
    PRED_SAVE_PATH,
    index=False
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print(
    "\n========== FINAL TRAINING SUMMARY =========="
)


print(
    "\nHeads trained:",
    len(saved_heads),
    "/ 5"
)


for seed in SEEDS:

    info = saved_heads[
        seed
    ]

    print(
        f"\nSeed {seed}:"
    )

    print(
        "  Selected config:",
        info["selected_config"]
    )

    print(
        "  Learning rate:",
        info["selected_learning_rate"]
    )

    print(
        "  Weight decay:",
        info["selected_weight_decay"]
    )

    print(
        "  Final epochs:",
        info["selected_epochs"]
    )


print(
    "\nRaw prediction rows:",
    len(predictions)
)

print(
    "Expected rows:",
    expected_rows
)

print(
    "NaN logits:",
    predictions[
        "raw_logit"
    ].isna().sum()
)

print(
    "Infinite logits:",
    np.isinf(
        predictions[
            "raw_logit"
        ].values
    ).sum()
)


print(
    "\nIMPORTANT:"
)

print(
    "Probability-calibration data was NOT used for HPO/training."
)

print(
    "Risk-calibration data was NOT used for HPO/training."
)

print(
    "Internal-test data was NOT used for HPO/training."
)

print(
    "Raw probabilities are still UNCALIBRATED."
)

print(
    "No selective threshold has been selected."
)

print(
    "No risk-control method has been applied."
)


print(
    "\nSaved final model bundle:"
)

print(
    MODEL_SAVE_PATH
)

print(
    "\nSaved raw predictions:"
)

print(
    PRED_SAVE_PATH
)

print(
    "\n========== COMPLETE =========="
)

Device: cuda

========== INPUT CHECK ==========
Embedding shape: (3504, 1024)
Unique image IDs: 3504
Manifest rows: 17520
Seeds: [42, 43, 44, 45, 46]

========== FINAL DINOV2-LARGE MLP TRAINING ==========

Backbone: DINOv2-Large
Encoder: FROZEN

Architecture:
1024 -> 512 -> ReLU -> Dropout(0.1) -> 1 logit

HPO:
3-fold inner stratified CV
Stratification: original grades 0-4
Selection metric: validation BCE loss

IMPORTANT:
HPO uses DEVELOPMENT data only.
Probability-calibration data: NOT USED
Risk-calibration data: NOT USED
Internal-test data: NOT USED


========== SEED 42 ==========
Development images: 2102
Non-referable: 1280
Referable: 822

Config A:
  Learning rate: 0.0003
  Weight decay: 0.0
  Fold best epochs: [14, 15, 24]
  Mean validation BCE: 0.159846
  Std validation BCE: 0.013312

Config B:
  Learning rate: 0.0003
  Weight decay: 0.0001
  Fold best epochs: [14, 15, 24]
  Mean validation BCE: 0.159810
  Std validation BCE: 0.013132

Config C:
  Learning rate: 0.001
  Weight de

#**DINOv2-Large PROBABILITY CALIBRATION**

In [5]:
# ============================================================
# DINOV2-LARGE PROBABILITY CALIBRATION
# SAME PROCEDURE AS RETFOUND-GREEN / MEDSIGLIP
#
# Uses ONLY:
#   prob_calibration split
#
# Saves:
#   aptos_dinov2_large_temperature_scaling.json
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.optimize import minimize_scalar
from scipy.special import expit


# ============================================================
# PATHS
# ============================================================

BASE = Path("/content/drive/MyDrive/thesis_DR")

PRED_PATH = (
    BASE /
    "aptos_dinov2_large_raw_predictions.csv"
)

SAVE_PATH = (
    BASE /
    "aptos_dinov2_large_temperature_scaling.json"
)


# ============================================================
# SETTINGS
# ============================================================

SEEDS = [42, 43, 44, 45, 46]
ECE_BINS = 10


# ============================================================
# LOAD
# ============================================================

pred = pd.read_csv(
    PRED_PATH,
    dtype={"image_id": str}
)

print("========== INPUT CHECK ==========")

print("Prediction rows:", len(pred))
print(
    "Seeds:",
    sorted(pred["seed"].unique().tolist())
)

assert len(pred) == 3504 * 5

assert sorted(
    pred["seed"].unique().tolist()
) == SEEDS

assert pred["raw_logit"].isna().sum() == 0

assert np.isfinite(
    pred["raw_logit"].values
).all()


# ============================================================
# METRICS
# ============================================================

def binary_nll_from_logits(
    logits,
    labels
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    labels = np.asarray(
        labels,
        dtype=np.float64
    )

    losses = (
        np.maximum(logits, 0)
        -
        logits * labels
        +
        np.log1p(
            np.exp(
                -np.abs(logits)
            )
        )
    )

    return float(
        np.mean(losses)
    )


def brier_score(
    probabilities,
    labels
):

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    labels = np.asarray(
        labels,
        dtype=np.float64
    )

    return float(
        np.mean(
            (
                probabilities
                -
                labels
            ) ** 2
        )
    )


def expected_calibration_error(
    probabilities,
    labels,
    n_bins=10
):

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    labels = np.asarray(
        labels,
        dtype=np.float64
    )

    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    ece = 0.0
    n = len(labels)

    for i in range(n_bins):

        lower = edges[i]
        upper = edges[i + 1]

        if i == n_bins - 1:

            mask = (
                (probabilities >= lower)
                &
                (probabilities <= upper)
            )

        else:

            mask = (
                (probabilities >= lower)
                &
                (probabilities < upper)
            )

        count = int(
            mask.sum()
        )

        if count == 0:
            continue

        confidence = float(
            probabilities[mask].mean()
        )

        observed_rate = float(
            labels[mask].mean()
        )

        ece += (
            count / n
        ) * abs(
            observed_rate
            -
            confidence
        )

    return float(ece)


# ============================================================
# FIT TEMPERATURE
# ============================================================

def fit_temperature(
    logits,
    labels
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    labels = np.asarray(
        labels,
        dtype=np.float64
    )

    def objective(
        log_temperature
    ):

        temperature = np.exp(
            log_temperature
        )

        return binary_nll_from_logits(
            logits / temperature,
            labels
        )

    result = minimize_scalar(
        objective,
        bounds=(-5.0, 5.0),
        method="bounded",
        options={
            "xatol": 1e-8
        }
    )

    temperature = float(
        np.exp(result.x)
    )

    original_nll = binary_nll_from_logits(
        logits,
        labels
    )

    optimized_nll = binary_nll_from_logits(
        logits / temperature,
        labels
    )

    if optimized_nll > original_nll:
        temperature = 1.0

    return temperature


# ============================================================
# FIT PER SEED
# ============================================================

results = {

    "model":
        "DINOv2-Large",

    "method":
        "single_parameter_temperature_scaling",

    "objective":
        "binary_negative_log_likelihood",

    "fit_split":
        "prob_calibration",

    "ece_bins":
        ECE_BINS,

    "seeds":
        {}
}


print(
    "\n========== DINOV2-LARGE TEMPERATURE SCALING =========="
)

print("\nFitting data: prob_calibration ONLY")
print("Risk calibration: NOT USED")
print("Internal test: NOT USED")


for seed in SEEDS:

    cal = pred[
        (
            pred["seed"] == seed
        )
        &
        (
            pred["split"]
            ==
            "prob_calibration"
        )
    ].copy()

    assert len(cal) == 350

    logits = cal[
        "raw_logit"
    ].to_numpy(
        dtype=np.float64
    )

    labels = cal[
        "binary_label"
    ].to_numpy(
        dtype=np.int64
    )


    # BEFORE
    raw_prob = expit(
        logits
    )

    nll_before = binary_nll_from_logits(
        logits,
        labels
    )

    brier_before = brier_score(
        raw_prob,
        labels
    )

    ece_before = expected_calibration_error(
        raw_prob,
        labels,
        ECE_BINS
    )


    # FIT T
    temperature = fit_temperature(
        logits,
        labels
    )


    # AFTER
    calibrated_logits = (
        logits / temperature
    )

    calibrated_prob = expit(
        calibrated_logits
    )

    nll_after = binary_nll_from_logits(
        calibrated_logits,
        labels
    )

    brier_after = brier_score(
        calibrated_prob,
        labels
    )

    ece_after = expected_calibration_error(
        calibrated_prob,
        labels,
        ECE_BINS
    )


    assert np.isfinite(
        temperature
    )

    assert temperature > 0

    assert (
        nll_after
        <=
        nll_before + 1e-10
    )


    results["seeds"][str(seed)] = {

        "temperature":
            float(temperature),

        "calibration_samples":
            int(len(cal)),

        "non_referable":
            int((labels == 0).sum()),

        "referable":
            int((labels == 1).sum()),

        "before": {
            "nll":
                float(nll_before),

            "brier":
                float(brier_before),

            "ece":
                float(ece_before)
        },

        "after": {
            "nll":
                float(nll_after),

            "brier":
                float(brier_after),

            "ece":
                float(ece_after)
        }
    }


    print(
        f"\n---------- SEED {seed} ----------"
    )

    print(
        "Calibration samples:",
        len(cal)
    )

    print(
        "Temperature:",
        f"{temperature:.6f}"
    )

    print(
        "NLL:",
        f"{nll_before:.6f}",
        "->",
        f"{nll_after:.6f}"
    )

    print(
        "Brier:",
        f"{brier_before:.6f}",
        "->",
        f"{brier_after:.6f}"
    )

    print(
        "ECE:",
        f"{ece_before:.6f}",
        "->",
        f"{ece_after:.6f}"
    )


# ============================================================
# SAVE
# ============================================================

with open(
    SAVE_PATH,
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )


# ============================================================
# SUMMARY
# ============================================================

temperatures = np.array([
    results[
        "seeds"
    ][str(seed)][
        "temperature"
    ]
    for seed in SEEDS
])


print(
    "\n========== FINAL SUMMARY =========="
)

print(
    "Mean temperature:",
    f"{temperatures.mean():.6f}"
)

print(
    "Temperature SD:",
    f"{temperatures.std(ddof=1):.6f}"
)

print(
    "\nSaved permanently:"
)

print(
    SAVE_PATH
)

print(
    "\nRisk calibration was NOT used."
)

print(
    "Internal test was NOT used."
)

print(
    "\n========== COMPLETE =========="
)

========== INPUT CHECK ==========
Prediction rows: 17520
Seeds: [42, 43, 44, 45, 46]

========== DINOV2-LARGE TEMPERATURE SCALING ==========

Fitting data: prob_calibration ONLY
Risk calibration: NOT USED
Internal test: NOT USED

---------- SEED 42 ----------
Calibration samples: 350
Temperature: 1.170562
NLL: 0.209326 -> 0.205869
Brier: 0.059526 -> 0.059024
ECE: 0.058918 -> 0.053970

---------- SEED 43 ----------
Calibration samples: 350
Temperature: 1.099681
NLL: 0.148714 -> 0.147821
Brier: 0.046644 -> 0.046118
ECE: 0.030500 -> 0.020903

---------- SEED 44 ----------
Calibration samples: 350
Temperature: 1.219520
NLL: 0.180126 -> 0.175703
Brier: 0.052813 -> 0.052194
ECE: 0.054631 -> 0.039503

---------- SEED 45 ----------
Calibration samples: 350
Temperature: 0.960203
NLL: 0.150257 -> 0.150100
Brier: 0.044877 -> 0.044958
ECE: 0.022694 -> 0.020398

---------- SEED 46 ----------
Calibration samples: 350
Temperature: 1.255672
NLL: 0.245687 -> 0.237656
Brier: 0.070830 -> 0.069563
ECE: 0.

#**DINOv2 source LTT Risk Control**

In [6]:
# ============================================================
# DINOV2-LARGE FIXED-SEQUENCE LTT RISK CONTROL
#
# EXACT SAME PROCEDURE AS RETFOUND / MEDSIGLIP
#
# Development:
#   candidate construction + ordering
#
# Risk calibration:
#   statistical certification
#
# Internal test:
#   NOT USED
#
# Saves:
#   aptos_dinov2_large_ltt_risk_control.csv
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.special import expit
from scipy.stats import binom


# ============================================================
# PATHS
# ============================================================

BASE = Path(
    "/content/drive/MyDrive/thesis_DR"
)

PRED_PATH = (
    BASE /
    "aptos_dinov2_large_raw_predictions.csv"
)

TEMP_PATH = (
    BASE /
    "aptos_dinov2_large_temperature_scaling.json"
)

SAVE_PATH = (
    BASE /
    "aptos_dinov2_large_ltt_risk_control.csv"
)


# ============================================================
# SETTINGS - SAME AS OTHER MODELS
# ============================================================

SEEDS = [42, 43, 44, 45, 46]

ALPHAS = [
    0.01,
    0.02,
    0.05
]

DELTA = 0.05

N_CANDIDATES = 50

QUANTILES = np.linspace(
    0.02,
    0.98,
    N_CANDIDATES
)


# ============================================================
# LOAD
# ============================================================

pred = pd.read_csv(
    PRED_PATH,
    dtype={"image_id": str}
)

with open(
    TEMP_PATH,
    "r"
) as f:

    temperature_data = json.load(f)


assert len(pred) == 3504 * 5

assert sorted(
    pred["seed"].unique().tolist()
) == SEEDS

assert pred[
    "raw_logit"
].isna().sum() == 0

assert np.isfinite(
    pred["raw_logit"].values
).all()


# ============================================================
# HOEFFDING-BENTKUS
# ============================================================

def binary_kl(x, mu):

    x = float(x)
    mu = float(mu)

    if x <= 0:

        return float(
            np.log(
                1.0 /
                (1.0 - mu)
            )
        )

    if x >= 1:

        return float(
            np.log(
                1.0 / mu
            )
        )

    return float(
        x * np.log(
            x / mu
        )
        +
        (1.0 - x)
        *
        np.log(
            (1.0 - x) /
            (1.0 - mu)
        )
    )


def hb_p_value(
    empirical_risk,
    n,
    alpha
):

    r_hat = float(
        empirical_risk
    )

    if r_hat >= alpha:
        return 1.0

    bentkus = (
        np.e
        *
        binom.cdf(
            int(
                np.ceil(
                    n * r_hat
                )
            ),
            n,
            alpha
        )
    )

    hoeffding = np.exp(
        -n
        *
        binary_kl(
            min(
                r_hat,
                alpha
            ),
            alpha
        )
    )

    return float(
        min(
            1.0,
            bentkus,
            hoeffding
        )
    )


# ============================================================
# THRESHOLD EVALUATION
# ============================================================

def evaluate_threshold(
    probabilities,
    labels,
    threshold,
    alpha
):

    auto = (
        probabilities
        <= threshold
    )

    n_auto = int(
        auto.sum()
    )

    errors = int(
        (
            auto
            &
            (labels == 1)
        ).sum()
    )

    coverage = (
        n_auto /
        len(labels)
    )

    review_burden = (
        1.0 -
        coverage
    )

    if n_auto > 0:

        observed_risk = (
            errors /
            n_auto
        )

    else:

        observed_risk = np.nan


    # Same transformed conditional-risk loss
    A = auto.astype(
        np.float64
    )

    losses = (
        A * labels
        +
        alpha * (
            1.0 - A
        )
    )

    empirical_ltt_loss = float(
        losses.mean()
    )


    return {

        "auto_clear_count":
            n_auto,

        "errors":
            errors,

        "coverage":
            float(
                coverage
            ),

        "review_burden":
            float(
                review_burden
            ),

        "observed_risk":
            float(
                observed_risk
            )
            if np.isfinite(
                observed_risk
            )
            else np.nan,

        "empirical_ltt_loss":
            empirical_ltt_loss
    }


# ============================================================
# RUN
# ============================================================

final_rows = []


print(
    "========== DINOV2-LARGE FIXED-SEQUENCE LTT =========="
)

print(
    "\nRisk targets:",
    ALPHAS
)

print(
    "Delta:",
    DELTA
)

print(
    "Candidate grid:",
    N_CANDIDATES
)

print(
    "Internal test used: NO"
)


for seed in SEEDS:

    temperature = float(
        temperature_data[
            "seeds"
        ][str(seed)][
            "temperature"
        ]
    )


    dev_df = pred[
        (
            pred["seed"] == seed
        )
        &
        (
            pred["split"]
            ==
            "development"
        )
    ].copy()


    risk_df = pred[
        (
            pred["seed"] == seed
        )
        &
        (
            pred["split"]
            ==
            "risk_calibration"
        )
    ].copy()


    assert len(dev_df) == 2102
    assert len(risk_df) == 526


    dev_prob = expit(
        dev_df[
            "raw_logit"
        ].to_numpy(
            dtype=np.float64
        )
        /
        temperature
    )


    risk_prob = expit(
        risk_df[
            "raw_logit"
        ].to_numpy(
            dtype=np.float64
        )
        /
        temperature
    )


    dev_labels = (
        dev_df[
            "binary_label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    risk_labels = (
        risk_df[
            "binary_label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    # ========================================================
    # SAME DEVELOPMENT QUANTILE GRID
    # ========================================================

    candidate_thresholds = np.unique(
        np.quantile(
            dev_prob,
            QUANTILES
        )
    )


    print(
        f"\n\n========== SEED {seed} =========="
    )

    print(
        "Development samples:",
        len(dev_df)
    )

    print(
        "Risk-calibration samples:",
        len(risk_df)
    )

    print(
        "Temperature:",
        f"{temperature:.6f}"
    )

    print(
        "Candidate thresholds:",
        len(candidate_thresholds)
    )


    for alpha in ALPHAS:

        # ====================================================
        # ORDER WITH DEVELOPMENT DATA ONLY
        # ====================================================

        ordering_rows = []


        for threshold in candidate_thresholds:

            dev_result = evaluate_threshold(
                dev_prob,
                dev_labels,
                threshold,
                alpha
            )

            ordering_rows.append({

                "threshold":
                    float(
                        threshold
                    ),

                "development_ltt_loss":
                    dev_result[
                        "empirical_ltt_loss"
                    ],

                "development_coverage":
                    dev_result[
                        "coverage"
                    ]
            })


        ordering_df = (
            pd.DataFrame(
                ordering_rows
            )
            .sort_values(
                [
                    "development_ltt_loss",
                    "development_coverage"
                ],
                ascending=[
                    True,
                    False
                ]
            )
            .reset_index(
                drop=True
            )
        )


        # ====================================================
        # FIXED-SEQUENCE TEST
        # ====================================================

        certified_candidates = []

        stop_p_value = np.nan
        stop_threshold = np.nan


        for position, row in ordering_df.iterrows():

            threshold = float(
                row[
                    "threshold"
                ]
            )


            result = evaluate_threshold(
                risk_prob,
                risk_labels,
                threshold,
                alpha
            )


            p_value = hb_p_value(
                result[
                    "empirical_ltt_loss"
                ],
                len(risk_labels),
                alpha
            )


            if (
                p_value <= DELTA
                and
                result[
                    "auto_clear_count"
                ] > 0
            ):

                certified_candidates.append({

                    "sequence_position":
                        int(
                            position + 1
                        ),

                    "threshold":
                        threshold,

                    "p_value":
                        float(
                            p_value
                        ),

                    **result
                })

            else:

                stop_p_value = float(
                    p_value
                )

                stop_threshold = float(
                    threshold
                )

                break


        # ====================================================
        # SELECT MAXIMUM-COVERAGE CERTIFIED POLICY
        # ====================================================

        if len(certified_candidates) > 0:

            certified_df = pd.DataFrame(
                certified_candidates
            )


            selected = (
                certified_df
                .sort_values(
                    [
                        "coverage",
                        "threshold"
                    ],
                    ascending=[
                        False,
                        True
                    ]
                )
                .iloc[0]
            )


            certified = True

            selected_threshold = float(
                selected[
                    "threshold"
                ]
            )

            selected_p_value = float(
                selected[
                    "p_value"
                ]
            )

            auto_clear_count = int(
                selected[
                    "auto_clear_count"
                ]
            )

            errors = int(
                selected[
                    "errors"
                ]
            )

            observed_risk = float(
                selected[
                    "observed_risk"
                ]
            )

            coverage = float(
                selected[
                    "coverage"
                ]
            )

            review_burden = float(
                selected[
                    "review_burden"
                ]
            )

            empirical_ltt_loss = float(
                selected[
                    "empirical_ltt_loss"
                ]
            )

            sequence_position = int(
                selected[
                    "sequence_position"
                ]
            )

        else:

            certified = False

            selected_threshold = np.nan
            selected_p_value = np.nan

            auto_clear_count = 0
            errors = 0

            observed_risk = np.nan

            coverage = 0.0

            review_burden = 1.0

            empirical_ltt_loss = np.nan

            sequence_position = np.nan


        final_rows.append({

            "seed":
                seed,

            "alpha":
                alpha,

            "delta":
                DELTA,

            "method":
                "Fixed_Sequence_LTT_Hoeffding_Bentkus",

            "ordering_source":
                "development_only",

            "candidate_grid_source":
                "development_probability_quantiles",

            "candidate_count":
                len(
                    candidate_thresholds
                ),

            "risk_calibration_n":
                len(
                    risk_df
                ),

            "certified":
                certified,

            "certified_candidates_before_stop":
                len(
                    certified_candidates
                ),

            "selected_sequence_position":
                sequence_position,

            "selected_threshold":
                selected_threshold,

            "selected_p_value":
                selected_p_value,

            "auto_clear_count":
                auto_clear_count,

            "errors":
                errors,

            "observed_auto_clear_risk":
                observed_risk,

            "coverage":
                coverage,

            "review_burden":
                review_burden,

            "empirical_ltt_loss":
                empirical_ltt_loss,

            "first_failed_threshold":
                stop_threshold,

            "first_failed_p_value":
                stop_p_value
        })


        print(
            f"\n--- Target alpha = {alpha:.2%} ---"
        )

        print(
            "Certified:",
            "YES"
            if certified
            else "NO"
        )

        print(
            "Candidates certified before stop:",
            len(
                certified_candidates
            )
        )


        if certified:

            print(
                "Selected threshold:",
                f"{selected_threshold:.6f}"
            )

            print(
                "Automatic non-referral count:",
                auto_clear_count
            )

            print(
                "Referable errors:",
                errors
            )

            print(
                "Observed non-referral risk:",
                f"{observed_risk:.4f}"
            )

            print(
                "Coverage:",
                f"{coverage:.4f}"
            )

            print(
                "Review burden:",
                f"{review_burden:.4f}"
            )

        else:

            print(
                "No threshold certified."
            )


# ============================================================
# SAVE
# ============================================================

results = pd.DataFrame(
    final_rows
)

assert len(results) == 15


results.to_csv(
    SAVE_PATH,
    index=False
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print(
    "\n\n========== FINAL DINOV2-LARGE LTT SUMMARY =========="
)


for alpha in ALPHAS:

    sub = results[
        np.isclose(
            results[
                "alpha"
            ],
            alpha
        )
    ]

    certified_sub = sub[
        sub[
            "certified"
        ]
    ]


    print(
        f"\nTarget alpha = {alpha:.2%}"
    )

    print(
        "Certified seeds:",
        len(
            certified_sub
        ),
        "/ 5"
    )


    if len(certified_sub) > 0:

        print(
            "Mean certified coverage:",
            f"{certified_sub['coverage'].mean():.4f}"
        )

        print(
            "Min certified coverage:",
            f"{certified_sub['coverage'].min():.4f}"
        )

        print(
            "Max certified coverage:",
            f"{certified_sub['coverage'].max():.4f}"
        )

        print(
            "Total risk-calibration referable errors:",
            int(
                certified_sub[
                    "errors"
                ].sum()
            )
        )


print(
    "\nSaved permanently:"
)

print(
    SAVE_PATH
)

print(
    "\nInternal test used: NO"
)

print(
    "\n========== COMPLETE =========="
)

========== DINOV2-LARGE FIXED-SEQUENCE LTT ==========

Risk targets: [0.01, 0.02, 0.05]
Delta: 0.05
Candidate grid: 50
Internal test used: NO


========== SEED 42 ==========
Development samples: 2102
Risk-calibration samples: 526
Temperature: 1.170562
Candidate thresholds: 50

--- Target alpha = 1.00% ---
Certified: NO
Candidates certified before stop: 0
No threshold certified.

--- Target alpha = 2.00% ---
Certified: NO
Candidates certified before stop: 0
No threshold certified.

--- Target alpha = 5.00% ---
Certified: NO
Candidates certified before stop: 0
No threshold certified.


========== SEED 43 ==========
Development samples: 2102
Risk-calibration samples: 526
Temperature: 1.099681
Candidate thresholds: 50

--- Target alpha = 1.00% ---
Certified: NO
Candidates certified before stop: 0
No threshold certified.

--- Target alpha = 2.00% ---
Certified: NO
Candidates certified before stop: 0
No threshold certified.

--- Target alpha = 5.00% ---
Certified: YES
Candidates certified be

#**DINOv2-Large result-summary**

In [7]:
# ============================================================
# DINOV2-LARGE COMPLETE SOURCE-DOMAIN RESULT SUMMARY
#
# Reads existing saved files only.
# Performs untouched internal-test evaluation.
#
# CREATES NO FILES.
#
# Reports:
#   1. Internal classification
#   2. Probability calibration
#   3. Source LTT certification
#   4. Held-out automatic non-referral
#   5. Final paper-level summary
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.special import expit
from scipy.stats import beta

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    balanced_accuracy_score
)


# ============================================================
# 1. PATHS
# ============================================================

BASE = Path(
    "/content/drive/MyDrive/thesis_DR"
)

PRED_FILE = (
    BASE /
    "aptos_dinov2_large_raw_predictions.csv"
)

TEMP_FILE = (
    BASE /
    "aptos_dinov2_large_temperature_scaling.json"
)

LTT_FILE = (
    BASE /
    "aptos_dinov2_large_ltt_risk_control.csv"
)


# ============================================================
# 2. SETTINGS
# ============================================================

SEEDS = [42, 43, 44, 45, 46]

PRIMARY_ALPHA = 0.05

CLASSIFICATION_THRESHOLD = 0.50


# ============================================================
# 3. LOAD EXISTING FILES
# ============================================================

pred = pd.read_csv(
    PRED_FILE,
    dtype={"image_id": str}
)

with open(
    TEMP_FILE,
    "r"
) as f:
    temp_data = json.load(f)

ltt = pd.read_csv(
    LTT_FILE
)


assert len(pred) == 3504 * 5

assert sorted(
    pred["seed"].unique().tolist()
) == SEEDS

assert pred[
    "raw_logit"
].isna().sum() == 0

assert np.isfinite(
    pred["raw_logit"].values
).all()


# ============================================================
# 4. HELD-OUT INTERNAL CLASSIFICATION
# ============================================================

classification_rows = []


for seed in SEEDS:

    test = pred[
        (
            pred["seed"] == seed
        )
        &
        (
            pred["split"] == "internal_test"
        )
    ].copy()

    assert len(test) == 526


    T = float(
        temp_data[
            "seeds"
        ][str(seed)][
            "temperature"
        ]
    )


    logits = test[
        "raw_logit"
    ].to_numpy(
        dtype=np.float64
    )


    labels = test[
        "binary_label"
    ].to_numpy(
        dtype=np.int64
    )


    probability = expit(
        logits / T
    )


    predicted = (
        probability
        >= CLASSIFICATION_THRESHOLD
    ).astype(int)


    tn, fp, fn, tp = confusion_matrix(
        labels,
        predicted,
        labels=[0, 1]
    ).ravel()


    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )


    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )


    ppv = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else np.nan
    )


    npv = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )


    classification_rows.append({

        "seed":
            seed,

        "n_test":
            len(test),

        "AUROC":
            roc_auc_score(
                labels,
                probability
            ),

        "AUPRC":
            average_precision_score(
                labels,
                probability
            ),

        "Sensitivity":
            sensitivity,

        "Specificity":
            specificity,

        "PPV":
            ppv,

        "NPV":
            npv,

        "F1":
            f1_score(
                labels,
                predicted
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                labels,
                predicted
            )
    })


classification_df = pd.DataFrame(
    classification_rows
)


# ============================================================
# 5. CALIBRATION SUMMARY
# ============================================================

calibration_rows = []


for seed in SEEDS:

    s = temp_data[
        "seeds"
    ][str(seed)]


    calibration_rows.append({

        "seed":
            seed,

        "temperature":
            s["temperature"],

        "NLL_before":
            s["before"]["nll"],

        "NLL_after":
            s["after"]["nll"],

        "Brier_before":
            s["before"]["brier"],

        "Brier_after":
            s["after"]["brier"],

        "ECE_before":
            s["before"]["ece"],

        "ECE_after":
            s["after"]["ece"]
    })


calibration_df = pd.DataFrame(
    calibration_rows
)


# ============================================================
# 6. PRIMARY 5% LTT POLICIES
# ============================================================

primary_ltt = ltt[
    np.isclose(
        ltt["alpha"],
        PRIMARY_ALPHA
    )
].copy()


assert len(primary_ltt) == 5


# ============================================================
# 7. HELD-OUT INTERNAL SELECTIVE-RISK EVALUATION
# ============================================================

risk_rows = []


for seed in SEEDS:

    test = pred[
        (
            pred["seed"] == seed
        )
        &
        (
            pred["split"] == "internal_test"
        )
    ].copy()


    T = float(
        temp_data[
            "seeds"
        ][str(seed)][
            "temperature"
        ]
    )


    probability = expit(
        test[
            "raw_logit"
        ].to_numpy(
            dtype=np.float64
        )
        /
        T
    )


    labels = test[
        "binary_label"
    ].to_numpy(
        dtype=np.int64
    )


    policy = primary_ltt[
        primary_ltt[
            "seed"
        ] == seed
    ].iloc[0]


    certified = bool(
        policy[
            "certified"
        ]
    )


    # ========================================================
    # FAIL-SAFE
    #
    # If no 5% source policy was certified,
    # no automatic non-referral is allowed.
    # ========================================================

    if certified:

        threshold = float(
            policy[
                "selected_threshold"
            ]
        )


        auto = (
            probability
            <= threshold
        )


        n_auto = int(
            auto.sum()
        )


        errors = int(
            (
                auto
                &
                (labels == 1)
            ).sum()
        )


        coverage = (
            n_auto /
            len(labels)
        )


        review_burden = (
            1.0 -
            coverage
        )


        if n_auto > 0:

            risk = (
                errors /
                n_auto
            )


            # One-sided 95% exact binomial upper bound
            # Evaluation uncertainty only
            if errors == n_auto:

                upper95 = 1.0

            else:

                upper95 = float(
                    beta.ppf(
                        0.95,
                        errors + 1,
                        n_auto - errors
                    )
                )

        else:

            risk = np.nan
            upper95 = np.nan


    else:

        threshold = np.nan

        n_auto = 0
        errors = 0

        coverage = 0.0
        review_burden = 1.0

        risk = np.nan
        upper95 = np.nan


    risk_rows.append({

        "seed":
            seed,

        "certified":
            certified,

        "threshold":
            threshold,

        "automatic_non_referral_n":
            n_auto,

        "referable_errors":
            errors,

        "coverage":
            coverage,

        "non_referral_risk":
            risk,

        "upper_95":
            upper95,

        "review_burden":
            review_burden,

        "observed_target_met":
            (
                bool(
                    risk <= PRIMARY_ALPHA
                )
                if np.isfinite(risk)
                else False
            ),

        "upper_bound_below_target":
            (
                bool(
                    upper95 <= PRIMARY_ALPHA
                )
                if np.isfinite(upper95)
                else False
            )
    })


risk_df = pd.DataFrame(
    risk_rows
)


# ============================================================
# 8. COMPLETE OUTPUT
# ============================================================

print(
    "=" * 78
)

print(
    "DINOV2-LARGE COMPLETE SOURCE-DOMAIN RESULTS"
)

print(
    "=" * 78
)


# ============================================================
# CLASSIFICATION
# ============================================================

print(
    "\n1. HELD-OUT INTERNAL CLASSIFICATION"
)

print(
    "-" * 78
)


print(
    classification_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.4f}"
    )
)


print(
    "\nMEAN ± SD"
)


metric_names = [
    "AUROC",
    "AUPRC",
    "Sensitivity",
    "Specificity",
    "PPV",
    "NPV",
    "F1",
    "Balanced_Accuracy"
]


for metric in metric_names:

    print(
        f"{metric:<20}"
        f"{classification_df[metric].mean():.4f}"
        f" ± "
        f"{classification_df[metric].std(ddof=1):.4f}"
    )


# ============================================================
# CALIBRATION
# ============================================================

print(
    "\n\n2. PROBABILITY CALIBRATION"
)

print(
    "-" * 78
)


print(
    calibration_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.4f}"
    )
)


print(
    "\nMEAN ± SD"
)


print(
    "NLL    "
    f"{calibration_df['NLL_before'].mean():.4f}"
    f" ± "
    f"{calibration_df['NLL_before'].std(ddof=1):.4f}"
    "  ->  "
    f"{calibration_df['NLL_after'].mean():.4f}"
    f" ± "
    f"{calibration_df['NLL_after'].std(ddof=1):.4f}"
)


print(
    "Brier  "
    f"{calibration_df['Brier_before'].mean():.4f}"
    f" ± "
    f"{calibration_df['Brier_before'].std(ddof=1):.4f}"
    "  ->  "
    f"{calibration_df['Brier_after'].mean():.4f}"
    f" ± "
    f"{calibration_df['Brier_after'].std(ddof=1):.4f}"
)


print(
    "ECE    "
    f"{calibration_df['ECE_before'].mean():.4f}"
    f" ± "
    f"{calibration_df['ECE_before'].std(ddof=1):.4f}"
    "  ->  "
    f"{calibration_df['ECE_after'].mean():.4f}"
    f" ± "
    f"{calibration_df['ECE_after'].std(ddof=1):.4f}"
)


# ============================================================
# LTT CERTIFICATION
# ============================================================

print(
    "\n\n3. SOURCE LTT CERTIFICATION"
)

print(
    "-" * 78
)


for alpha in [
    0.01,
    0.02,
    0.05
]:

    sub = ltt[
        np.isclose(
            ltt["alpha"],
            alpha
        )
    ]

    print(
        f"{100*alpha:.0f}% target: "
        f"{int(sub['certified'].sum())}/5 seeds certified"
    )


# ============================================================
# INTERNAL NON-REFERRAL
# ============================================================

print(
    "\n\n4. HELD-OUT INTERNAL AUTOMATIC NON-REFERRAL"
)

print(
    "-" * 78
)


print(
    risk_df[
        [
            "seed",
            "certified",
            "threshold",
            "automatic_non_referral_n",
            "referable_errors",
            "coverage",
            "non_referral_risk",
            "upper_95",
            "review_burden",
            "observed_target_met",
            "upper_bound_below_target"
        ]
    ].to_string(
        index=False,
        float_format=lambda x:
            f"{x:.4f}"
    )
)


# ============================================================
# FINAL PAPER-LEVEL SUMMARY
# ============================================================

print(
    "\n\n5. FINAL DINOV2-LARGE SUMMARY"
)

print(
    "=" * 78
)


print(
    "\nCLASSIFICATION"
)


for metric in metric_names:

    print(
        f"{metric:<20}"
        f"{classification_df[metric].mean():.4f}"
        f" ± "
        f"{classification_df[metric].std(ddof=1):.4f}"
    )


print(
    "\nCALIBRATION"
)


print(
    f"NLL:   "
    f"{calibration_df['NLL_before'].mean():.4f}"
    " -> "
    f"{calibration_df['NLL_after'].mean():.4f}"
)


print(
    f"Brier: "
    f"{calibration_df['Brier_before'].mean():.4f}"
    " -> "
    f"{calibration_df['Brier_after'].mean():.4f}"
)


print(
    f"ECE:   "
    f"{calibration_df['ECE_before'].mean():.4f}"
    " -> "
    f"{calibration_df['ECE_after'].mean():.4f}"
)


print(
    "\n5% SOURCE RISK CONTROL"
)


print(
    "LTT certification:"
    f" {int(risk_df['certified'].sum())}/5 seeds"
)


print(
    "Operational automatic non-referral coverage:"
    f" {100 * risk_df['coverage'].mean():.2f}"
    f" ± "
    f"{100 * risk_df['coverage'].std(ddof=1):.2f}%"
)


print(
    "Operational review burden:"
    f" {100 * risk_df['review_burden'].mean():.2f}"
    f" ± "
    f"{100 * risk_df['review_burden'].std(ddof=1):.2f}%"
)


evaluated = risk_df[
    risk_df[
        "certified"
    ]
].copy()


if len(evaluated) > 0:

    print(
        "Observed non-referral risk among certified runs:"
        f" {100 * evaluated['non_referral_risk'].mean():.2f}"
        f" ± "
        f"{100 * evaluated['non_referral_risk'].std(ddof=1):.2f}%"
    )


    print(
        "One-sided 95% upper risk bound:"
        f" {100 * evaluated['upper_95'].mean():.2f}"
        f" ± "
        f"{100 * evaluated['upper_95'].std(ddof=1):.2f}%"
    )


    print(
        "5% observed-risk target satisfied:"
        f" {int(evaluated['observed_target_met'].sum())}"
        f"/{len(evaluated)} evaluated certified runs"
    )


    print(
        "95% upper bound below 5%:"
        f" {int(evaluated['upper_bound_below_target'].sum())}"
        f"/{len(evaluated)} evaluated certified runs"
    )


    print(
        "Total held-out referable errors:"
        f" {int(evaluated['referable_errors'].sum())}"
    )


uncertified = risk_df[
    ~risk_df[
        "certified"
    ]
]["seed"].tolist()


print(
    "\nSeed(s) with no certified 5% source policy:"
)

print(
    uncertified
    if len(uncertified) > 0
    else "None"
)


print(
    "\nNo files created or modified by this cell."
)

print(
    "=" * 78
)

DINOV2-LARGE COMPLETE SOURCE-DOMAIN RESULTS

1. HELD-OUT INTERNAL CLASSIFICATION
------------------------------------------------------------------------------
 seed  n_test  AUROC  AUPRC  Sensitivity  Specificity    PPV    NPV     F1  Balanced_Accuracy
   42     526 0.9801 0.9643       0.9659       0.9065 0.8684 0.9765 0.9145             0.9362
   43     526 0.9809 0.9660       0.9073       0.9346 0.8986 0.9404 0.9029             0.9209
   44     526 0.9787 0.9599       0.9512       0.9128 0.8744 0.9670 0.9112             0.9320
   45     526 0.9781 0.9576       0.9515       0.9062 0.8673 0.9667 0.9074             0.9289
   46     526 0.9861 0.9736       0.9951       0.8812 0.8436 0.9965 0.9131             0.9382

MEAN ± SD
AUROC               0.9808 ± 0.0032
AUPRC               0.9643 ± 0.0062
Sensitivity         0.9542 ± 0.0317
Specificity         0.9083 ± 0.0190
PPV                 0.8705 ± 0.0196
NPV                 0.9694 ± 0.0202
F1                  0.9098 ± 0.0047
Balanced_Accu

| Metric                                |      RETFound-Green |           MedSigLIP |        DINOv2-Large | Best         |
| ------------------------------------- | ------------------: | ------------------: | ------------------: | ------------ |
| **AUROC**                             |     0.9800 ± 0.0018 | **0.9822 ± 0.0018** |     0.9808 ± 0.0032 | MedSigLIP    |
| **AUPRC**                             |     0.9648 ± 0.0051 | **0.9677 ± 0.0042** |     0.9643 ± 0.0062 | MedSigLIP    |
| **Sensitivity**                       |     0.9211 ± 0.0276 |     0.9425 ± 0.0397 | **0.9542 ± 0.0317** | DINOv2       |
| **Specificity**                       | **0.9276 ± 0.0191** |     0.9257 ± 0.0148 |     0.9083 ± 0.0190 | RETFound     |
| **PPV**                               | **0.8917 ± 0.0232** |     0.8913 ± 0.0165 |     0.8705 ± 0.0196 | RETFound     |
| **NPV**                               |     0.9488 ± 0.0165 |     0.9625 ± 0.0237 | **0.9694 ± 0.0202** | DINOv2       |
| **F1**                                |     0.9057 ± 0.0072 | **0.9155 ± 0.0132** |     0.9098 ± 0.0047 | MedSigLIP    |
| **Balanced accuracy**                 |     0.9244 ± 0.0071 | **0.9341 ± 0.0138** |     0.9312 ± 0.0068 | MedSigLIP    |
| **NLL after calibration**             |     0.1768 ± 0.0071 | **0.1489 ± 0.0156** |     0.1834 ± 0.0383 | MedSigLIP    |
| **Brier after calibration**           |     0.0539 ± 0.0016 | **0.0450 ± 0.0064** |     0.0544 ± 0.0102 | MedSigLIP    |
| **ECE after calibration**             |     0.0293 ± 0.0074 | **0.0277 ± 0.0069** |     0.0442 ± 0.0273 | MedSigLIP    |
| **5% LTT certification**              |             **5/5** |                 4/5 |                 3/5 | **RETFound** |
| **Operational non-referral coverage** |   **51.06 ± 1.61%** |      43.76 ± 24.49% |      32.05 ± 29.27% | **RETFound** |
| **Review burden**                     |   **48.94 ± 1.61%** |      56.24 ± 24.49% |      67.95 ± 29.27% | **RETFound** |
| **Observed non-referral risk**        |    **0.37 ± 0.36%** |       0.95 ± 0.57%* |       0.83 ± 0.19%* | **RETFound** |
| **One-sided 95% upper risk bound**    |    **1.71 ± 0.57%** |       2.51 ± 0.79%* |       2.39 ± 0.25%* | **RETFound** |
| **Uncertified seeds at 5%**           |            **None** |                  44 |              42, 46 | **RETFound** |


What this tells us

There is now a very clear and interesting pattern.

MedSigLIP is the strongest conventional classifier. It has the best AUROC, AUPRC, F1, balanced accuracy, NLL, Brier, and ECE.

DINOv2 has the highest sensitivity and NPV, but its calibration is poorer and its risk-control policy is unstable.

RETFound-Green is clearly the strongest model for our actual deployment objective. It is the only one with:

5/5 LTT certification + approximately 51% coverage + only 0.37% observed non-referral risk + stable coverage across seeds.

This is scientifically useful because it shows that:

The model with the best conventional classification metrics is not necessarily the model with the strongest statistically controlled selective-deployment behavior.